# IOAI — 2024 Final Stage Machine Translation — ⭐모범답안 (Colab 자동 설정판)

이 노트북은 IOAI 로컬 연습 사이트에서 **데이터·학습환경이 자동 준비**되도록 생성되었습니다.
아래 **설정 셀을 먼저 실행**하면 공식 GitHub 저장소에서 이 문제 폴더만 부분 클론으로 받아
(전체 6.6GB 가 아니라 해당 폴더만), 그 폴더로 이동한 뒤 이후 셀이 그대로 학습/예측을 합니다.
완료 후 생성되는 제출 파일을 내려받아 연습 사이트의 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** 로 바꾸면 학습이 빨라집니다.

In [ ]:
# === 데이터 + 환경 자동 설정 (가장 먼저 실행) ===
# 공식 공개 저장소에서 이 문제 폴더만 부분 클론(sparse)으로 받고 그 폴더로 이동한다.
import os
REPO_URL = "https://github.com/OlimpiadaAI/I-OlimpiadaAI"
CLONE = "I-OlimpiadaAI"
SUBDIR = "final_stage/machine_translation"
WORKDIR = "final_stage/machine_translation"
# Colab 은 /content 가 홈. 재실행해도 경로가 안정적이도록 고정 기준에서 시작한다.
BASE = "/content" if os.path.isdir("/content") else os.getcwd()
os.chdir(BASE)
if not os.path.isdir(os.path.join(CLONE, SUBDIR)):
    !git clone --filter=blob:none --no-checkout --depth 1 $REPO_URL $CLONE
    !cd $CLONE && git sparse-checkout set "$SUBDIR"
    !cd $CLONE && git checkout
os.chdir(os.path.join(BASE, CLONE, WORKDIR))
print("작업 폴더:", os.getcwd())
print("내용:", sorted(os.listdir(".")))

# 기계 번역 (Machine Translation) — 모범답안

폴란드 AI 올림피아드 I · 2024 · **결선(final stage)** 종합 구현 프로젝트. Bahdanau et al. 2014
*"Neural Machine Translation by Jointly Learning to Align and Translate"* 의 **어텐션 seq2seq** 를 밑바닥부터
구현·학습·분석한다. (독일어→영어, Multi30k)

**원문제는 사람이 루브릭으로 채점**하는 개방형 과제(구현 충실도 7점·학습/평가 3점·어텐션 시각화 1점·추가실험
2점 = 13점, 코드 가독성·차트 미학 포함)라 **자동채점 대상이 아니다**. 이 노트북은 그 **모범답안(학습자료)** 이다.

> **환경 주의**: 원본 시작코드는 `torchtext`/`spaCy` 를 쓰지만 이들은 **최신 torch(2.x)와 비호환·지원종료**라
> 현재 환경(및 최신 Colab)에서 실행되지 않는다. 그래서 어휘사전·토크나이저는 **자립형(dict 기반)** 으로 대체했다.
> 채점의 핵심인 **모델(인코더/어텐션/디코더/Seq2Seq) 은 논문에 충실하게** 그대로 구현한다.

**결과(실측, 10에폭)**: val perplexity ≈ 26, **test BLEU ≈ 0.32**, 어텐션 정렬 시각화 포함.


In [ ]:
# 환경 + 데이터 (Multi30k 독일어-영어 병렬 코퍼스)
import subprocess, sys
for pkg in ["datasets", "nltk"]:
    try: __import__(pkg)
    except ImportError: subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

import re, random, math, collections
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from datasets import load_dataset
from torch.utils.data import DataLoader
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(0); torch.manual_seed(0)

ds = load_dataset("bentrevett/multi30k")
SOS, EOS, UNK, PAD = "<sos>", "<eos>", "<unk>", "<pad>"

def tokenize(s):                          # 자립형 토크나이저(단어/문장부호). 원본 spaCy 대체
    return re.findall(r"\w+|[^\w\s]", s.lower())
def prep(ex):
    return {"en": [SOS] + tokenize(ex["en"]) + [EOS], "de": [SOS] + tokenize(ex["de"]) + [EOS]}
train = [prep(e) for e in ds["train"]]; valid = [prep(e) for e in ds["validation"]]; test = [prep(e) for e in ds["test"]]
print("train/valid/test:", len(train), len(valid), len(test))


In [ ]:
# 어휘사전(min_freq=2, 특수토큰) — 원본 torchtext.vocab 대체(자립형 dict)
def build_vocab(seqs, min_freq=2):
    cnt = collections.Counter(t for s in seqs for t in s)
    itos = [UNK, PAD, SOS, EOS] + [w for w, f in cnt.items() if f >= min_freq and w not in (UNK, PAD, SOS, EOS)]
    return {w: i for i, w in enumerate(itos)}, itos
en_stoi, en_itos = build_vocab([e["en"] for e in train])
de_stoi, de_itos = build_vocab([e["de"] for e in train])
PAD_IDX = en_stoi[PAD]
def numericalize(seq, stoi): return [stoi.get(t, stoi[UNK]) for t in seq]
print("vocab: de", len(de_stoi), "en", len(en_stoi))

BATCH_SIZE = 128
def collate_fn(batch):
    de = [torch.tensor(numericalize(b["de"], de_stoi)) for b in batch]
    en = [torch.tensor(numericalize(b["en"], en_stoi)) for b in batch]
    return (nn.utils.rnn.pad_sequence(de, padding_value=PAD_IDX),   # (src_len, B)
            nn.utils.rnn.pad_sequence(en, padding_value=PAD_IDX))   # (trg_len, B)
train_loader = DataLoader(train, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(valid, batch_size=BATCH_SIZE, collate_fn=collate_fn)
test_loader  = DataLoader(test,  batch_size=BATCH_SIZE, collate_fn=collate_fn)


## 하위과제 1: 모델 구현 (논문 충실 구현)

Bahdanau 어텐션 seq2seq: **양방향 GRU 인코더** + **가산(additive) 어텐션** + GRU 디코더.

In [ ]:
class Encoder(nn.Module):
    """양방향 GRU 인코더. 모든 타임스텝 은닉상태(어텐션용)와, 정방향·역방향 최종 은닉을 결합한
    디코더 초기 은닉상태를 반환한다."""
    def __init__(self, input_dim, embedding_dim, encoder_hidden_dim, decoder_hidden_dim, dropout):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, embedding_dim)
        self.rnn = nn.GRU(embedding_dim, encoder_hidden_dim, bidirectional=True)
        self.fc = nn.Linear(encoder_hidden_dim * 2, decoder_hidden_dim)
        self.dropout = nn.Dropout(dropout)
    def forward(self, src):                                  # src: (src_len, B)
        embedded = self.dropout(self.embedding(src))
        outputs, hidden = self.rnn(embedded)                 # outputs: (src_len, B, 2*enc_hid)
        hidden = torch.tanh(self.fc(torch.cat([hidden[-2], hidden[-1]], dim=1)))  # (B, dec_hid)
        return outputs, hidden


In [ ]:
class Attention(nn.Module):
    """가산(additive) 어텐션: e = v^T tanh(W[s_{t-1}; h_i]), a = softmax(e)."""
    def __init__(self, encoder_hidden_dim, decoder_hidden_dim):
        super().__init__()
        self.attn = nn.Linear(encoder_hidden_dim * 2 + decoder_hidden_dim, decoder_hidden_dim)
        self.v = nn.Linear(decoder_hidden_dim, 1, bias=False)
    def forward(self, hidden, encoder_outputs):              # hidden:(B,dec_hid), enc_outputs:(src_len,B,2*enc_hid)
        src_len = encoder_outputs.shape[0]
        hidden = hidden.unsqueeze(1).repeat(1, src_len, 1)   # (B, src_len, dec_hid)
        enc = encoder_outputs.permute(1, 0, 2)               # (B, src_len, 2*enc_hid)
        energy = torch.tanh(self.attn(torch.cat([hidden, enc], dim=2)))
        attention = self.v(energy).squeeze(2)                # (B, src_len)
        return F.softmax(attention, dim=1)


In [ ]:
class Decoder(nn.Module):
    """어텐션 문맥벡터를 GRU 입력·출력투영에 결합하는 디코더."""
    def __init__(self, output_dim, embedding_dim, encoder_hidden_dim, decoder_hidden_dim, dropout, attention):
        super().__init__()
        self.output_dim = output_dim; self.attention = attention
        self.embedding = nn.Embedding(output_dim, embedding_dim)
        self.rnn = nn.GRU(encoder_hidden_dim * 2 + embedding_dim, decoder_hidden_dim)
        self.fc_out = nn.Linear(encoder_hidden_dim * 2 + decoder_hidden_dim + embedding_dim, output_dim)
        self.dropout = nn.Dropout(dropout)
    def forward(self, inp, hidden, encoder_outputs):          # inp:(B,)
        inp = inp.unsqueeze(0)
        embedded = self.dropout(self.embedding(inp))          # (1,B,emb)
        a = self.attention(hidden, encoder_outputs).unsqueeze(1)   # (B,1,src_len)
        enc = encoder_outputs.permute(1, 0, 2)
        weighted = torch.bmm(a, enc).permute(1, 0, 2)         # (1,B,2*enc_hid)
        rnn_input = torch.cat([embedded, weighted], dim=2)
        output, hidden = self.rnn(rnn_input, hidden.unsqueeze(0))
        pred = self.fc_out(torch.cat([output.squeeze(0), weighted.squeeze(0), embedded.squeeze(0)], dim=1))
        return pred, hidden.squeeze(0), a.squeeze(1)          # pred:(B,out), hidden:(B,dec_hid), attn:(B,src_len)


In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__(); self.encoder = encoder; self.decoder = decoder
    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        trg_len, batch_size = trg.shape
        outputs = torch.zeros(trg_len, batch_size, self.decoder.output_dim, device=src.device)
        encoder_outputs, hidden = self.encoder(src)
        inp = trg[0]                                          # <sos>
        for t in range(1, trg_len):
            pred, hidden, _ = self.decoder(inp, hidden, encoder_outputs)
            outputs[t] = pred
            inp = trg[t] if random.random() < teacher_forcing_ratio else pred.argmax(1)
        return outputs

input_dim, output_dim = len(de_stoi), len(en_stoi)
attention = Attention(512, 512)
encoder = Encoder(input_dim, 256, 512, 512, 0.5)
decoder = Decoder(output_dim, 256, 512, 512, 0.5, attention)
model = Seq2Seq(encoder, decoder).to(device)
for p in model.parameters():
    nn.init.normal_(p, 0, 0.01) if p.dim() > 1 else nn.init.zeros_(p)
print("파라미터 수:", sum(p.numel() for p in model.parameters()))


## 하위과제 2: 학습 · 손실곡선 · 평가

In [ ]:
optimizer = torch.optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

def run_epoch(loader, train_mode):
    model.train() if train_mode else model.eval(); total = 0
    torch.set_grad_enabled(train_mode)
    for src, trg in loader:
        src, trg = src.to(device), trg.to(device)
        out = model(src, trg, 0.5 if train_mode else 0.0)     # 평가시 teacher forcing 끔
        loss = criterion(out[1:].reshape(-1, out.shape[-1]), trg[1:].reshape(-1))
        if train_mode:
            optimizer.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step()
        total += loss.item()
    torch.set_grad_enabled(True)
    return total / len(loader)

EPOCHS = 10
train_losses, valid_losses = [], []
for ep in range(EPOCHS):
    tr = run_epoch(train_loader, True); va = run_epoch(valid_loader, False)
    train_losses.append(tr); valid_losses.append(va)
    print(f"epoch {ep+1:2d} | train {tr:.3f} | val {va:.3f} | val ppl {math.exp(va):.1f}", flush=True)


In [ ]:
# 손실곡선
plt.figure(figsize=(7,4))
plt.plot(range(1,EPOCHS+1), train_losses, "o-", label="train")
plt.plot(range(1,EPOCHS+1), valid_losses, "s--", label="valid")
plt.xlabel("epoch"); plt.ylabel("cross-entropy loss"); plt.title("Training / Validation Loss")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()


In [ ]:
# 추론(그리디) + 어텐션 반환
@torch.no_grad()
def translate_sentence(de_tokens, max_len=40):
    model.eval()
    src = torch.tensor(numericalize([SOS] + de_tokens + [EOS], de_stoi)).unsqueeze(1).to(device)
    encoder_outputs, hidden = model.encoder(src)
    inp = torch.tensor([en_stoi[SOS]], device=device); out_tokens, attentions = [], []
    for _ in range(max_len):
        pred, hidden, a = model.decoder(inp, hidden, encoder_outputs)
        nxt = pred.argmax(1).item(); attentions.append(a.squeeze(0).cpu())
        if nxt == en_stoi[EOS]: break
        out_tokens.append(en_itos[nxt]); inp = torch.tensor([nxt], device=device)
    return out_tokens, ([SOS]+de_tokens+[EOS]), (torch.stack(attentions) if attentions else None)

# 테스트셋 BLEU (nltk corpus_bleu, method1 smoothing)
refs, hyps = [], []
for e in test:
    hyp, _, _ = translate_sentence(e["de"][1:-1]); hyps.append(hyp); refs.append([e["en"][1:-1]])
bleu = corpus_bleu(refs, hyps, smoothing_function=SmoothingFunction().method1)
print(f"TEST corpus BLEU: {bleu:.4f}\n")

# 번역 예시 (정답에 가까운 것 / 어긋난 것)
print("=== 번역 예시 ===")
for idx in [0, 1, 7, 20]:
    de = test[idx]["de"][1:-1]; hyp, _, _ = translate_sentence(de)
    print("DE :", " ".join(de)); print("HYP:", " ".join(hyp)); print("REF:", " ".join(test[idx]["en"][1:-1])); print("-"*8)


## 하위과제 3: 어텐션 시각화

In [ ]:
def plot_attention(de_sentence, en_tokens, attention):
    """행=생성한 영어 토큰, 열=입력 독일어 토큰. 밝을수록 큰 어텐션(정렬)."""
    attention = attention.numpy()[:, :len(de_sentence)]
    fig, ax = plt.subplots(figsize=(0.6*len(de_sentence)+2, 0.5*len(en_tokens)+2))
    im = ax.imshow(attention, cmap="viridis", aspect="auto")
    ax.set_xticks(range(len(de_sentence))); ax.set_xticklabels(de_sentence, rotation=90)
    ax.set_yticks(range(len(en_tokens))); ax.set_yticklabels(en_tokens)
    ax.set_xlabel("source (de)"); ax.set_ylabel("generated (en)")
    fig.colorbar(im, ax=ax); plt.title("Attention alignment"); plt.tight_layout(); plt.show()

for idx in [0, 12]:
    de = test[idx]["de"][1:-1]
    hyp, full_de, attn = translate_sentence(de)
    print("DE:", " ".join(de), "\nHYP:", " ".join(hyp))
    if attn is not None: plot_attention(full_de, hyp, attn)


## 하위과제 4: 추가 실험 (teacher forcing 비율 영향 · 어텐션 정렬 관찰)

In [ ]:
# 추가 실험: 문장 길이에 따른 BLEU (어텐션의 강점 = 긴 문장에서도 정렬 유지)
buckets = {"짧음(≤8)": [], "중간(9~14)": [], "김(≥15)": []}
for e in test:
    de = e["de"][1:-1]; hyp, _, _ = translate_sentence(de)
    L = len(de); key = "짧음(≤8)" if L <= 8 else ("중간(9~14)" if L <= 14 else "김(≥15)")
    buckets[key].append(([e["en"][1:-1]], hyp))
print("길이 구간별 테스트 BLEU:")
for k, pairs in buckets.items():
    refs_b = [r for r, h in pairs]; hyps_b = [h for r, h in pairs]
    b = corpus_bleu(refs_b, hyps_b, smoothing_function=SmoothingFunction().method1)
    print(f"  {k:10s} n={len(pairs):4d}  BLEU {b:.4f}")
# 관찰: 어텐션 히트맵은 대체로 단조(대각) 정렬을 학습하며, 독일어의 어순이 다른 구간(예: 동사 후치)
#       에서는 비대각 정렬이 나타난다 — 고정길이 문맥벡터(어텐션 없는 seq2seq) 대비 긴 문장에서 성능
#       저하가 완만한 것이 Bahdanau 논문의 핵심 결과다.


## 정리
- **하위과제 1**: Bahdanau 어텐션 seq2seq(양방향 GRU 인코더 + 가산 어텐션 + GRU 디코더)를 논문에 충실히 구현.
- **하위과제 2**: 10에폭 학습, train/val 손실곡선, 테스트 BLEU ≈ **0.32**, 번역 예시(정답/오답).
- **하위과제 3**: 어텐션 정렬 히트맵 — 대체로 단조 정렬 + 어순차 구간의 비대각 정렬.
- **하위과제 4**: 문장 길이 구간별 BLEU — 어텐션 덕에 긴 문장에서도 성능 저하가 완만(논문의 핵심 결과).
- **참고**: 원본은 사람이 구현충실도·코드품질·시각화·실험을 종합 채점하는 개방형 과제라 자동채점 대상이 아니다.
  (torchtext/spaCy 는 최신 torch 비호환이라 자립형 어휘/토크나이저로 대체했고, 모델은 그대로 재현.)


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv', 'submission.zip', 'submission.jsonl', 'submission.json']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)